# QHED (Quantum Hadamard Edge Detection)
QHED (Quantum Hadamard Edge Detection) sits at the intersection of Quantum Image Processing (QIP) and Quantum Machine Learning.

The major bottleneck in classical image processing is that filtering an image requires sweeping a kernel across pixels sequentially, yielding a complexity of $O(N^2)$ for an $N \times N$ image. QHED uses amplitude encoding to compress an entire grayscale image of $2^n$ pixels into the quantum amplitudes of just $n$ qubits. By applying a single Hadamard gate to an auxiliary qubit, the quantum computer calculates the gradient between all adjacent pixels simultaneously via quantum interference.

For a true research-level setup, we won't just detect edges statically. We will construct a hybrid QHED-Driven Quantum Image Classifier Pipeline. The pipeline works by using QHED as a quantum pre-processing feature extractor layer that extracts structural contours, followed by a trainable parameterized quantum circuit (PQC) that classifies the images based on those structural features.The

##Hybrid Pipeline Architecture
###Step 1: Synthesizing the Visual Dataset
We will create a clean $4 \times 4$ image dataset consisting of two classes that are easy to distinguish by their edge profiles: Vertical Lines (Class 0) and Horizontal Lines (Class 1).

In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# Create 4x4 images (16 pixels total -> fits perfectly into 4 qubits: 2^4 = 16)
# Class 0: Vertical stripes
img_vertical = np.array([
    [1, 0, 1, 0],
    [1, 0, 1, 0],
    [1, 0, 1, 0],
    [1, 0, 1, 0]
]).flatten()

# Class 1: Horizontal stripes
img_horizontal = np.array([
    [1, 1, 1, 1],
    [0, 0, 0, 0],
    [1, 1, 1, 1],
    [0, 0, 0, 0]
]).flatten()

# Normalize amplitudes so their squares sum to 1 (Required for quantum states)
img_vertical = img_vertical / np.linalg.norm(img_vertical)
img_horizontal = img_horizontal / np.linalg.norm(img_horizontal)

dataset = [img_vertical, img_horizontal]
labels = [0, 1]  # 0 for Vertical, 1 for Horizontal

###Step 2: Designing the QHED + Classifier Circuit
We need 5 qubits total:

Qubits 1, 2, 3, 4 represent the 16 pixel positions of our image (data_wires).

Qubit 0 is an auxiliary processing qubit (aux_wire) used to calculate pixel differences through interference.

In [ ]:
num_data_qubits = 4
total_qubits = num_data_qubits + 1  # 1 aux qubit + 4 image data qubits

dev = qml.device("braket.local.qubit", wires=total_qubits)

@qml.qnode(dev)
def qhed_classifier_circuit(image_state, classification_weights):
    # --- PHASE 1: AMPLITUDE ENCODING ---
    # Load the 16 normalized pixel intensities directly into the 4 data qubits
    qml.AmplitudeEmbedding(features=image_state, wires=range(1, total_qubits))

    # --- PHASE 2: QUANTUM HADAMARD EDGE DETECTION (QHED) ---
    # 1. Put the auxiliary qubit into superposition
    qml.Hadamard(wires=0)

    # 2. Apply a decrement operation (Shift operator) conditioned on the aux qubit.
    # This shifts the pixel indexes by 1 to align neighboring pixels side-by-side.
    # For a simulation research setup, we can use a unitary permutation matrix.
    N = 2**num_data_qubits
    permutation_matrix = np.zeros((N, N))
    for i in range(N):
        permutation_matrix[(i + 1) % N, i] = 1.0

    qml.ControlledQubitUnitary(permutation_matrix, control_wires=[0], wires=range(1, total_qubits))

    # 3. Apply the final Hadamard to cause interference between the original and shifted image
    qml.Hadamard(wires=0)

    # --- PHASE 3: PARAMETERIZED QUANTUM CLASSIFIER ---
    # The system state now contains the edge data. We feed this into a
    # strongly entangling ansatz to train the classification boundaries.
    qml.StronglyEntanglingLayers(weights=classification_weights, wires=range(total_qubits))

    # Measure expectation to assign a binary classification label (-1 to +1)
    return qml.expval(qml.PauliZ(total_qubits - 1))

###Step 3: Training the Pipeline
We initialize our hybrid classifier and optimize the parameters using classical gradient descent to see if the network can accurately identify the image profiles.

In [ ]:
# Setup ansatz dimensions: 2 layers, 5 total qubits, 3 rotation parameters per gate
ansatz_layers = 2
np.random.seed(42)
weights = np.random.uniform(0, 2 * np.pi, (ansatz_layers, total_qubits, 3), requires_grad=True)

opt = qml.AdamOptimizer(stepsize=0.1)

# Map labels from [0, 1] to quantum expectation values [-1, 1]
target_expectations = np.array([-1.0, 1.0])

print("Training QHED Image Classifier...")
print("---------------------------------")

for epoch in range(26):
    loss = 0
    for img, label_idx in zip(dataset, [0, 1]):
        target = target_expectations[label_idx]

        # Mean Squared Error Loss Function
        def cost_fn(w):
            prediction = qhed_classifier_circuit(img, w)
            return (prediction - target) ** 2

        weights = opt.step(cost_fn, weights)
        loss += cost_fn(weights)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d} | Total Square Loss: {loss:.5f}")

###Step 4: Verification & Performance Metrics

In [ ]:
print("\n--- Model Evaluation ---")
for i, img in enumerate(dataset):
    raw_prediction = qhed_classifier_circuit(img, weights)
    # Map expectation back to class labels
    predicted_class = 1 if raw_prediction > 0 else 0
    true_class = labels[i]

    class_name = "Vertical Line" if true_class == 0 else "Horizontal Line"
    print(f"Image [{class_name}] -> Raw Expectation: {raw_prediction:7.4f} | Predicted Class: {predicted_class} (Match: {predicted_class == true_class})")

##Advancing This Setup Into a Research Paper
If you to scale this code framework into a proper peer-reviewed publication, we should explore these structural extensions:

**Robustness Against Noise and Compression Blurs**
Real images captured by optical sensors contain environmental noise and compression artifacts. A strong research paper could benchmark this pipeline against traditional classical filters (like Sobel or Canny edge detection). Students can add varying levels of Gaussian noise to the input matrix (img_vertical + noise) and document the threshold where the QHED state encoding degrades compared to classical preprocessing methods.

**Algorithmic Complexity of State Preparation**
While the Hadamard operation takes $O(1)$ time execution on a QPU, loading an arbitrary classical image into qml.AmplitudeEmbedding requires an initialization step that scales as $O(2^n)$ gates classically. This is known as the State Preparation Problem in QML. A highly valuable contribution involves designing or evaluating alternate sub-exponential state preparation schemes (like FRQI—Flexible Representation of Quantum Images) specifically within an Amazon Braket pipeline.

**Edge-Preserving Quantum Variational Autoencoders (QVAE)**
Instead of a simple classifier, you can link this QHED feature extractor to a Quantum Variational Autoencoder. The network's job would be to compress an image into a low-dimensional quantum bottleneck state, using the QHED layer as a regularizer in the loss function to force the autoencoder to preserve sharp geometric boundaries during reconstruction.